In [79]:
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv('rps_data.csv', sep=',')

In [80]:
df['opp_match_wins'] = pd.to_numeric(df['opp_match_wins'], errors='coerce').fillna(-1).astype(int)
df['opp_match_winrate'] = pd.to_numeric(df['opp_match_winrate'], errors='coerce').fillna(-1.0)
df['score_diff'] = df['score_me_before'] - df['score_opp_before']
df['prev2_opp_move'] = df.groupby('match_id')['opp_move'].shift(2)
df['prev2_opp_move'] = df['prev2_opp_move'].fillna('-1')
df['next_opp_move'] = df.groupby('match_id')['opp_move'].shift(-1)
df = df.dropna(subset=['next_opp_move']).copy()

In [81]:
df.info()

<class 'pandas.DataFrame'>
Index: 133 entries, 0 to 156
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   match_id           133 non-null    int64  
 1   round              133 non-null    int64  
 2   opp_match_wins     133 non-null    int64  
 3   opp_match_winrate  133 non-null    float64
 4   stake              133 non-null    int64  
 5   opp_move           133 non-null    str    
 6   my_move            133 non-null    str    
 7   outcome            133 non-null    str    
 8   score_me_before    133 non-null    int64  
 9   score_opp_before   133 non-null    int64  
 10  prev_opp_move      133 non-null    str    
 11  prev_outcome       133 non-null    str    
 12  streak_draws       133 non-null    int64  
 13  score_diff         133 non-null    int64  
 14  prev2_opp_move     133 non-null    str    
 15  next_opp_move      133 non-null    str    
dtypes: float64(1), int64(8), str(7)
memory usa

In [82]:
df

,match_id,round,opp_match_wins,opp_match_winrate,stake,opp_move,my_move,outcome,score_me_before,score_opp_before,prev_opp_move,prev_outcome,streak_draws,score_diff,prev2_opp_move,next_opp_move
0,1,1,-1,-1.00,25,Н,Б,lose,0,0,-1,none,0,0,-1,К
1,1,2,-1,-1.00,25,К,Н,lose,0,1,Н,lose,0,-1,-1,Н
2,1,3,-1,-1.00,25,Н,Н,draw,0,2,К,lose,0,-2,Н,Н
3,1,4,-1,-1.00,25,Н,Н,draw,0,2,Н,draw,1,-2,К,Н
4,1,5,-1,-1.00,25,Н,К,win,0,2,Н,draw,2,-2,Н,Н
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152,25,1,24,54.55,25,К,К,draw,0,0,-1,none,0,0,-1,К
153,25,2,24,54.55,25,К,Б,win,0,0,К,draw,1,0,-1,К
154,25,3,24,54.55,25,К,К,draw,1,0,К,win,0,1,К,Н
155,25,4,24,54.55,25,Н,К,win,1,0,К,draw,1,1,К,Б


In [83]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

le_target = LabelEncoder()
df['next_opp_move_enc'] = le_target.fit_transform(df['next_opp_move'])

# Определяем признаки (X) и целевую (y)
categorical_features = ['opp_move', 'my_move', 'outcome', 'prev_opp_move', 'prev_outcome', 'prev2_opp_move']
numeric_features = ['opp_match_wins', 'opp_match_winrate', 'stake', 
                    'score_me_before', 'score_opp_before', 'streak_draws']

X = df[categorical_features + numeric_features]
y = df['next_opp_move_enc']

# Разделяем на обучение и тест (стратификация по y)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [84]:
# Препроцессор: OneHot для категорий, StandardScaler для чисел (хотя Random Forest не требует, но оставим)
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features),
    ('num', 'passthrough', numeric_features)   # без масштабирования, можно добавить StandardScaler()
])

# Модель Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=42,
    class_weight='balanced'
)

# Пайплайн
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', rf)
])

In [85]:
# Обучение
pipeline.fit(X_train, y_train)

# Предсказание на тесте
y_pred = pipeline.predict(X_test)

# Оценка
acc = accuracy_score(y_test, y_pred)
print(f'Accuracy: {acc:.3f}')
print(classification_report(y_test, y_pred, target_names=le_target.classes_))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.481
              precision    recall  f1-score   support

           Б       0.50      0.25      0.33         8
           К       0.47      0.70      0.56        10
           Н       0.50      0.44      0.47         9

    accuracy                           0.48        27
   macro avg       0.49      0.46      0.45        27
weighted avg       0.49      0.48      0.46        27

Confusion matrix:
[[2 3 3]
 [2 7 1]
 [0 5 4]]


In [86]:
joblib.dump(pipeline, 'rps_model.pkl')
joblib.dump(le_target, 'target_encoder.pkl')

['target_encoder.pkl']